# 06 — Сравнение экспериментов и сборка итоговых таблиц/графиков

Этот ноутбук НЕ запускает оптимизацию. Он берёт собранные в `results/raw/*.csv`
сводки (после прогона 01–05) и строит:
- единую таблицу `tables/experiments_summary.csv`;
- сравнение метрик `tables/metrics_comparison.xlsx`;
- графики в `figures/`.

> Если `results/raw` пуст — сначала прогоните ноутбуки 01–05 (там ячейка `run_original_and_harvest`).

In [ ]:
# === Единая настройка путей и окружения (общая для всех wrapper-ноутбуков) ===
# ВАЖНО: этот блок НЕ трогает исходные ноутбуки. Он только готовит окружение,
# монтирует Drive и определяет, где лежат (а) исходные .ipynb и (б) папка-репозиторий tpe.
import os, sys, json, shutil, subprocess, time, glob
from pathlib import Path

# 1) Монтируем Google Drive (в Colab). Локально просто пропустится.
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print('Drive mount skipped:', repr(e))

# 2) Где лежат ИСХОДНЫЕ ноутбуки (5 файлов fin_*.ipynb).
#    Поменяйте при необходимости на свой путь в Drive.
ORIGINALS_DIR = Path(os.environ.get('ORIGINALS_DIR', 'drive/MyDrive/content/k5_originals'))
if not ORIGINALS_DIR.exists():
    for cand in ['/content/drive/MyDrive/content/k5_originals',
                 'drive/MyDrive/content', '/content/drive/MyDrive/content', '.']:
        if list(Path(cand).glob('fin_*.ipynb')):
            ORIGINALS_DIR = Path(cand); break

# 3) Корень анализа (эта папка coursework_analysis).
ANALYSIS_ROOT = Path(os.environ.get('ANALYSIS_ROOT', '.')).resolve()
for cand in [Path('.'), Path('coursework_analysis'), Path('/content/coursework_analysis')]:
    if (cand / 'configs' / 'common_config.json').exists():
        ANALYSIS_ROOT = cand.resolve(); break

RAW_DIR   = ANALYSIS_ROOT / 'results' / 'raw'
PROC_DIR  = ANALYSIS_ROOT / 'results' / 'processed'
TABLES_DIR= ANALYSIS_ROOT / 'tables'
FIG_DIR   = ANALYSIS_ROOT / 'figures'
for d in (RAW_DIR, PROC_DIR, TABLES_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

COMMON = json.load(open(ANALYSIS_ROOT / 'configs' / 'common_config.json'))
EXPS   = json.load(open(ANALYSIS_ROOT / 'configs' / 'experiments_config.json'))
EXP_BY_ID = {e['id']: e for e in EXPS['experiments']}

print('ORIGINALS_DIR =', ORIGINALS_DIR)
print('ANALYSIS_ROOT =', ANALYSIS_ROOT)
print('originals found:', sorted(p.name for p in ORIGINALS_DIR.glob('fin_*.ipynb')))

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

def load(name):
    p = RAW_DIR / name
    return pd.read_csv(p) if p.exists() else None

# Унифицируем сводки 03/04/05 (одинаковая схема колонок) в одну длинную таблицу.
frames = []
for fid, fname in [('03_raw_vs_norm','03_raw_vs_norm_summary.csv'),
                   ('04_tpe_wx','04_tpe_wx_summary_visible.csv'),
                   ('05_gtpe','05_gtpe_summary_visible.csv')]:
    df = load(fname)
    if df is not None:
        df = df.copy(); df['experiment'] = fid
        frames.append(df)
big = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
print('rows:', len(big))
big.head()

In [ ]:
# Главная таблица: file -> experiment -> вариант -> режим -> метрики
if len(big):
    cols = ['experiment','function','variant','scale_type','data_type',
            'success_rate_%','steps_mean','final_dist_x_mean','final_dist_y_mean']
    cols = [c for c in cols if c in big.columns]
    summary = big[cols].sort_values(cols[:5])
    summary.to_csv(TABLES_DIR / 'experiments_summary.csv', index=False)
    print('saved tables/experiments_summary.csv')
    display(summary.head(30))

In [ ]:
# Сравнение метрик в Excel: по листу на функцию.
if len(big):
    xlsx = TABLES_DIR / 'metrics_comparison.xlsx'
    try:
        with pd.ExcelWriter(xlsx) as xw:
            big.to_excel(xw, sheet_name='all', index=False)
            for fn in big['function'].unique():
                big[big['function']==fn].to_excel(xw, sheet_name=str(fn)[:28], index=False)
        print('saved', xlsx)
    except Exception as e:
        print('xlsx skipped (нет openpyxl?):', repr(e))
        big.to_csv(TABLES_DIR / 'metrics_comparison.csv', index=False)

## Прямое сравнение метода 2 (w(x)) с baseline `no_w` — единственный корректный ablation

In [ ]:
# Для каждой (function, scale, data): no_w против лучшего grad_* по final_dist_y_mean.
df4 = load('04_tpe_wx_summary_visible.csv')
if df4 is not None:
    rows=[]
    for (fn,sc,dt), g in df4.groupby(['function','scale_type','data_type']):
        base = g[g['variant']=='no_w']
        grads = g[g['variant'].str.startswith('grad_')]
        if len(base) and len(grads):
            b = float(base['final_dist_y_mean'].iloc[0])
            best = grads.loc[grads['final_dist_y_mean'].idxmin()]
            rows.append({'function':fn,'scale':sc,'data':dt,
                         'no_w_dist_y':b,'best_grad':best['variant'],
                         'best_grad_dist_y':float(best['final_dist_y_mean']),
                         'grad_better':bool(best['final_dist_y_mean']<b)})
    abl = pd.DataFrame(rows)
    abl.to_csv(TABLES_DIR / 'ablation_wx_vs_no_w.csv', index=False)
    display(abl)
    print('grad_* лучше no_w в', int(abl['grad_better'].sum()), 'из', len(abl), 'настроек')

In [ ]:
# График: success_rate по вариантам для выбранной функции/режима (если данные есть).
if len(big) and 'success_rate_%' in big.columns:
    sub = big[(big['function']=='Ackley(2D)') & (big.get('data_type','clean')=='clean') & (big.get('scale_type','raw')=='raw')]
    if len(sub):
        ax = sub.groupby('variant')['success_rate_%'].mean().sort_values().plot.barh(figsize=(8,5))
        ax.set_title('Ackley(2D), raw, clean — success_rate по вариантам')
        plt.tight_layout(); plt.savefig(FIG_DIR / 'ackley_raw_clean_success.png', dpi=130); plt.show()